# NaMaster vs GMaster: methods trust from shipped artifacts

This notebook **loads committed demo outputs only** — no new MASTER or NaMaster runs on ACT or FLAMINGO maps.

Each case directory contains five files: `bins.npy`, `cl_gmaster.npy`, `cl_namaster.npy`, `overlay.pdf`, `README.txt`.

**Methods trust only.** We compare decoupled TT bandpowers and wall-clock from saved runs. This is not a cosmological analysis.

- **ACT DR6:** native HEALPix `nside=8192`; demo uses `healpy.ud_grade` to `nside=4096` with the survey footprint mask (`f_sky ≈ 0.48`).
- **FLAMINGO L2p8 Compton-y:** full-sky `nside=4096` (`f_sky = 1`).

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

EXAMPLES = Path(__file__).resolve().parent if "__file__" in dir() else Path("examples")
if not (EXAMPLES / "act_dr6_tt_example").exists():
    EXAMPLES = Path(".").resolve()

CASES = {
    "ACT DR6 TT": EXAMPLES / "act_dr6_tt_example",
    "FLAMINGO L2p8 y": EXAMPLES / "flamingo_y_tt_example",
}


def load_case(case_dir: Path) -> dict:
    ell = np.load(case_dir / "bins.npy")
    cl_gm = np.load(case_dir / "cl_gmaster.npy")
    cl_nm = np.load(case_dir / "cl_namaster.npy")
    readme = (case_dir / "README.txt").read_text()
    return {"ell": ell, "cl_gm": cl_gm, "cl_nm": cl_nm, "readme": readme}


def parse_readme(text: str) -> dict:
    out = {}
    m = re.search(r"t_gmaster=([0-9.]+)s", text)
    out["t_gmaster_s"] = float(m.group(1)) if m else np.nan
    m = re.search(r"t_namaster=([0-9.]+)s", text)
    out["t_namaster_s"] = float(m.group(1)) if m else np.nan
    m = re.search(r"cores=([0-9]+)", text)
    out["namaster_cores"] = int(m.group(1)) if m else None
    m = re.search(r"rms=([0-9.eE+-]+)", text)
    out["rms_ratio"] = float(m.group(1)) if m else np.nan
    m = re.search(r"max\|\.\|=([0-9.eE+-]+)", text)
    out["max_ratio"] = float(m.group(1)) if m else np.nan
    m = re.search(r"nside=([0-9]+)", text)
    out["nside"] = int(m.group(1)) if m else None
    return out


data = {name: load_case(path) for name, path in CASES.items()}
meta = {name: parse_readme(d["readme"]) for name, d in data.items()}
list(CASES.keys())

In [ ]:
header = (
    f"{'case':<18} {'nside':>5} {'rms(GM/NM-1)':>14} {'t_gm (s)':>10} "
    f"{'t_nm (s)':>10} {'cores':>5} {'NM/GM':>8}"
)
print(header)
print("-" * len(header))
for name, m in meta.items():
    speedup = m["t_namaster_s"] / m["t_gmaster_s"] if m["t_gmaster_s"] else float("nan")
    print(
        f"{name:<18} {m['nside']:>5} {m['rms_ratio']:>14.3e} {m['t_gmaster_s']:>10.3f} "
        f"{m['t_namaster_s']:>10.3f} {m['namaster_cores']:>5} {speedup:>8.1f}x"
    )

In [ ]:
BLUE, GREY, BLACK = "#0072B2", "#666666", "#000000"


def plot_overlay(name: str, case_dir: Path, d: dict, m: dict) -> None:
    ell, cl_gm, cl_nm = d["ell"], d["cl_gm"], d["cl_nm"]
    ratio = cl_gm / cl_nm - 1.0
    dl = ell * (ell + 1) / (2 * np.pi)

    fig, ax = plt.subplots(2, 1, sharex=True, figsize=(7.16, 4.6),
                           gridspec_kw={"height_ratios": [2.2, 1]})
    ax[0].plot(ell, dl * cl_gm, color=BLUE, lw=1.35,
               label=rf"GMaster ${m['t_gmaster_s']:.2f}\,\mathrm{{s}}$")
    ax[0].plot(ell, dl * cl_nm, color=GREY, lw=1.15, ls="--",
               label=rf"NaMaster ({m['namaster_cores']} cores) ${m['t_namaster_s']:.1f}\,\mathrm{{s}}$")
    ax[0].set_ylabel(r"$D_\ell=\ell(\ell+1)C_\ell/2\pi$")
    ax[0].set_yscale("log")
    ax[0].set_title(name)
    ax[0].legend(frameon=False)

    ax[1].axhline(0, color=BLACK, lw=0.6)
    ax[1].plot(ell, 1e5 * ratio, color=BLUE, lw=0.9)
    ax[1].set_ylabel(r"$(C_\ell^{\mathrm{GM}}/C_\ell^{\mathrm{NM}}-1)\times 10^{5}$")
    ax[1].set_xlabel(r"$\ell$")
    fig.tight_layout()
    plt.show()

    overlay_pdf = case_dir / "overlay.pdf"
    if overlay_pdf.exists():
        print(f"Committed overlay: {overlay_pdf}")


for name, case_dir in CASES.items():
    plot_overlay(name, case_dir, data[name], meta[name])